## Model components

In [13]:
from pyomo.environ import *
import numpy as np
import math
# import pyomo.environ as pyo

## Creat master problem

In [14]:
scenarios = [1, 2]
prob = {1: 0.4, 2: 0.6}
d = {1: {1: 500, 2: 300}, 2: {1: 100, 2: 300}}
q = {1: {1: -24, 2: -28}, 2:{1: -28, 2: -32}}


    # Master Problem
model = ConcreteModel()
model.I = RangeSet(2)
model.x = Var(model.I, domain=NonNegativeReals)
model.theta = Var(domain=Reals)    

model.obj = Objective(expr=100*model.x[1] + 150*model.x[2] + model.theta)

model.con1 = Constraint(expr=model.x[1] + model.x[2] <= 120)
model.con2 = Constraint(expr=model.x[1] >= 40)
model.con3 = Constraint(expr=model.x[2] >= 20)


# H&T

In [15]:
from pyomo.environ import ConcreteModel, Var, NonNegativeReals, Objective, Constraint, SolverFactory, value
T = {}
h = {}

for s in scenarios:

    sub = ConcreteModel()
    sub.y = Var(model.I, within=NonNegativeReals)
    sub.obj = Objective(expr=
                         q[1][s]*sub.y[1] + q[2][s]*sub.y[2],
                           sense=minimize)

    sub.c1 = Constraint(expr = 6*sub.y[1] + 10*sub.y[2] <= 60 * (model.x[1]))
    sub.c2 = Constraint(expr = 8*sub.y[1] + 5*sub.y[2] <= 80 * (model.x[2]))
    sub.c3 = Constraint(expr = sub.y[1] <= d[1][s])
    sub.c4 = Constraint(expr = sub.y[2] <= d[2][s])

    solver = SolverFactory('cplex')

    #ضرایب T

    body1 = sub.c1.body
    body2 = sub.c2.body
    body3 = sub.c3.body
    body4 = sub.c4.body
    
    

    term_T = [body1, body2, body3, body4]

    def get_coefficients(expression, variable):
        if hasattr(expression, 'args'):
            for term in expression.args:
                if term.is_expression_type() and len(term.args) == 2:
                    coef, var = term.args
                    if var is variable:
                        return coef
        return None
    
    T_tmp = []
    T_tmp.append([get_coefficients(body1, model.x[1]),0])
    T_tmp.append([0,get_coefficients(body2, model.x[2])])
    T_tmp.append([0,0]) # no x in third constraint
    T_tmp.append([0,0]) # no x in fourth constraint
    

    T[s] = T_tmp

    #ضرایب h

    def get_rhs(constraint):
        if constraint.has_lb() and not constraint.has_ub():
            return constraint.lower
        elif constraint.has_ub() and not constraint.has_lb():
            return constraint.upper
        elif constraint.has_lb() and constraint.has_ub():
            if constraint.lower == constraint.upper:
                return constraint.lower
        return None
    h_tmp = []
    h_tmp.append(float(get_rhs(sub.c1)))  
    h_tmp.append(float(get_rhs(sub.c2)))
    h_tmp.append(float(get_rhs(sub.c3)))
    h_tmp.append(float(get_rhs(sub.c4)))
      
    h[s] = h_tmp
    


        

In [16]:
T


{1: [[-60, 0], [0, -80], [0, 0], [0, 0]],
 2: [[-60, 0], [0, -80], [0, 0], [0, 0]]}

In [17]:
h

{1: [0.0, 0.0, 500.0, 100.0], 2: [0.0, 0.0, 300.0, 300.0]}

## Multi-Cut L-Shape algorithm

In [18]:
# start
r, s, v = (0, 0, 0)
theta_values = []
w_values = []

#Step 1: Define Master problem
model = ConcreteModel()
model.I = RangeSet(2)
model.x = Var(model.I, domain=NonNegativeReals)
model.theta = Var(scenarios, domain=Reals)
model.obj = Objective(expr=100*model.x[1] + 150*model.x[2] + model.theta[1] + model.theta[2])

model.con1 = Constraint(expr=model.x[1] + model.x[2] <= 120)
model.con2 = Constraint(expr=model.x[1] >= 40)
model.con3 = Constraint(expr=model.x[2] >= 20)
model.optimal_cut = ConstraintList()

not_stop_condition = True
while not_stop_condition:
    not_stop_conditionF = True
    while not_stop_conditionF:

        v += 1
        if v == 1:
            model.cons4 = Constraint(expr = model.theta[1] == -10000000)
            model.cons5 = Constraint(expr = model.theta[2] == -10000000)   
            theta_values.append(-10000000) #teta1
            theta_values.append(-10000000) #teta2
        else:
            model.cons4.deactivate()
            model.cons5.deactivate()
            print(f"\\n[Theta from previous iteration: {value(model.theta[s]):,.2f}]")
            theta_values.append(value(model.theta[s]))

        solver = SolverFactory('cplex')
        result = solver.solve(model, tee=False)

        print("\n" + "="*50) 
        print(f"🔁 Iteration {v}")   
        print("-"*50) 
        print(f"📌 Master Problem Objective Value : {value(model.obj):,.2f}")  
        print(f"📌 Decision Variables:")
        print(f"    x1 = {value(model.x[1]):.2f}")
        print(f"    x2 = {value(model.x[2]):.2f}")
        print("="*50)


        #step2
        di=[]
        D=[]
        simplex_multiplierF = {}

        for s in scenarios:
            feas = ConcreteModel()
            feas.y = Var(model.I, within=NonNegativeReals)
            feas.v_plus = Var(RangeSet(4), domain=NonNegativeReals)
            feas.v_minus = Var(RangeSet(4), domain=NonNegativeReals)

            feas.obj = Objective(expr=sum(feas.v_plus[i] + feas.v_minus[i] for i in RangeSet(4)), sense=minimize)

            feas.c1 = Constraint(expr = 6*feas.y[1] + 10*feas.y[2] + feas.v_plus[1] - feas.v_minus[1] <= 60 * value(model.x[1]))
            feas.c2 = Constraint(expr = 8*feas.y[1] + 5*feas.y[2] + feas.v_plus[2] - feas.v_minus[2] <= 80 * value(model.x[2]))
            feas.c3 = Constraint(expr = feas.y[1] + feas.v_plus[3] - feas.v_minus[3] <= d[1][s])
            feas.c4 = Constraint(expr = feas.y[2] + feas.v_plus[4] - feas.v_minus[4] <= d[2][s])
            feas.dual = Suffix(direction=Suffix.IMPORT)

            solver = SolverFactory('cplex')
            result = solver.solve(feas, tee=False)

            dual_var_F=[]
            for c in feas.component_objects( Constraint, active = True ):
                cobject = getattr( feas, str( c ) )
                for index in cobject:
                    dual_var_F.append(feas.dual[cobject[index]])

            simplex_multiplierF[s] = dual_var_F

            print(f"\n🔹 Scenario {s}:")
            
            print(f"    π (Dual values) = {simplex_multiplierF[s]}")          
        
            di.append(prob[s] * np.array([simplex_multiplierF[s]]) @ np.array(h[s]))
            D.append(prob[s] * np.array([simplex_multiplierF[s]]) @ np.array(T[s]))

        print("\n🧮 Aggregated Values:")
        print(f"    di = {di}")
        print(f"    D = {D}")
        print('---'*10)

        zero = 0 
        feasibility_violated = any(value(feas.obj) > zero for _ in scenarios)


        if feasibility_violated:
            r += 1
            model.optimal_cut.add(list(D @ np.array([model.x[1], model.x[2]]))[0] + model.theta[s] >= di[0])   
            print("\n✅ Added optimality cut to master problem.")
            model.optimal_cut.pprint()
            print("="*50 + "\n")
        else:
            print("\n🛑 Feasibility is OK!")
            
            print("="*50 + "\n")
            not_stop_conditionF = False


    #step3

    e = []
    E = []
    simplex_multiplier = {}

    for s in scenarios:
        sub = ConcreteModel()
        sub.y = Var(model.I, within=NonNegativeReals)
        sub.obj = Objective(expr=
                        q[1][s]*sub.y[1] + q[2][s]*sub.y[2],
                        sense=minimize)

        sub.c1 = Constraint(expr = 6*sub.y[1] + 10*sub.y[2] <= 60 * value(model.x[1]))
        sub.c2 = Constraint(expr = 8*sub.y[1] + 5*sub.y[2] <= 80 * value(model.x[2]))
        sub.c3 = Constraint(expr = sub.y[1] <= d[1][s])
        sub.c4 = Constraint(expr = sub.y[2] <= d[2][s])
        sub.dual = Suffix(direction=Suffix.IMPORT)

        solver = SolverFactory('cplex')
        result = solver.solve(sub, tee=False)

        dual_var_2=[]
        for c in sub.component_objects( Constraint, active = True ):
            cobject = getattr( sub, str( c ) )
            for index in cobject:
                dual_var_2.append(sub.dual[cobject[index]])

        simplex_multiplier[s] = dual_var_2
        print(f"    simplex_multiplier = {simplex_multiplier[s]}")



        print(f"\n🔹 Scenario {s}:")
        print(f"    y1 = {value(sub.y[1]):.2f}, y2 = {value(sub.y[2]):.2f}")  
        print(f"    π (Dual values) = {simplex_multiplier[s]}")          
            
        e.append(prob[s] * np.array([simplex_multiplier[s]]) @ np.array(h[s]))
        E.append(prob[s] * np.array([simplex_multiplier[s]]) @ np.array(T[s]))

    print("\n🧮 Aggregated Values:")
    print(f"    e = {e}")
    print(f"    e1 = {e[0]}")
    print(f"    e2 = {e[1]}")
    print(f"    E = {E}")
    print(f"    E2 = {E[1]}")
    print('---'*10)
    print(f"    E1 = {E[0]}")



        

    for s in scenarios:
        w = h[s] - T[s] @ np.array([value(model.x[1]), value(model.x[2])])
        dot_product = []
        dot_product = prob[s] * np.dot(simplex_multiplier[s], w)

        if dot_product > value(model.theta[s]):
            x_values = [model.x[1], model.x[2]]

            model.optimal_cut.add(list(E[s-1] @ np.array([model.x[1], model.x[2]]))[0] + model.theta[s] >= float(e[s-1]))

            print("\n✅ Added optimality cut to master problem.")
            model.optimal_cut.pprint()
            print("="*50 + "\n")
        else:
            solver = SolverFactory('cplex')
            result = solver.solve(model, tee=False)

            print("\n" + "="*50) 
            print(f"🔁 Iteration {v}")   
            print("-"*50) 
            print(f"📌 Master Problem Objective Value : {value(model.obj):,.2f}")  
            print(f"📌 Decision Variables:")
            print(f"    x1 = {value(model.x[1]):.2f}")
            print(f"    x2 = {value(model.x[2]):.2f}")
            print("="*50)
            print("\n🛑 Termination Criteria Met!")
            print(f"✅ Final Master Problem Objective: {value(model.obj):,.2f}")
            print(f"✅ Final Decision Variables: x1 = {value(model.x[1]):.2f}, x2 = {value(model.x[2]):.2f}")
            print("="*50 + "\n")
            print(f"✅ Final counter Variables: r = {value(r)},s= {value(s)}, v= {value(v)}")
            not_stop_condition = False


🔁 Iteration 1
--------------------------------------------------
📌 Master Problem Objective Value : -19,993,000.00
📌 Decision Variables:
    x1 = 40.00
    x2 = 20.00

🔹 Scenario 1:
    π (Dual values) = [0.0, 0.0, 0.0, 0.0]

🔹 Scenario 2:
    π (Dual values) = [0.0, 0.0, 0.0, 0.0]

🧮 Aggregated Values:
    di = [array([0.]), array([0.])]
    D = [array([[0., 0.]]), array([[0., 0.]])]
------------------------------

🛑 Feasibility is OK!

    simplex_multiplier = [0.0, -3.0, 0.0, -13.0]

🔹 Scenario 1:
    y1 = 137.50, y2 = 100.00
    π (Dual values) = [0.0, -3.0, 0.0, -13.0]
    simplex_multiplier = [-2.3200000000000003, -1.7600000000000002, 0.0, 0.0]

🔹 Scenario 2:
    y1 = 80.00, y2 = 192.00
    π (Dual values) = [-2.3200000000000003, -1.7600000000000002, 0.0, 0.0]

🧮 Aggregated Values:
    e = [array([-520.]), array([0.])]
    e1 = [-520.]
    e2 = [0.]
    E = [array([[ 0., 96.]]), array([[83.52, 84.48]])]
    E2 = [[83.52 84.48]]
------------------------------
    E1 = [[ 0. 96.]]

C:\Users\mmm\AppData\Local\Temp\ipykernel_19400\2183453065.py:168: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  model.optimal_cut.add(list(E[s-1] @ np.array([model.x[1], model.x[2]]))[0] + model.theta[s] >= float(e[s-1]))



🔹 Scenario 1:
    π (Dual values) = [0.0, 0.0, 0.0, 0.0]

🔹 Scenario 2:
    π (Dual values) = [0.0, 0.0, 0.0, 0.0]

🧮 Aggregated Values:
    di = [array([0.]), array([0.])]
    D = [array([[0., 0.]]), array([[0., 0.]])]
------------------------------

🛑 Feasibility is OK!

    simplex_multiplier = [-4.0, 0.0, 0.0, 0.0]

🔹 Scenario 1:
    y1 = 400.00, y2 = 0.00
    π (Dual values) = [-4.0, 0.0, 0.0, 0.0]
    simplex_multiplier = [-3.2, 0.0, -8.799999999999997, 0.0]

🔹 Scenario 2:
    y1 = 300.00, y2 = 60.00
    π (Dual values) = [-3.2, 0.0, -8.799999999999997, 0.0]

🧮 Aggregated Values:
    e = [array([0.]), array([-1584.])]
    e1 = [0.]
    e2 = [-1584.]
    E = [array([[96.,  0.]]), array([[115.2,   0. ]])]
    E2 = [[115.2   0. ]]
------------------------------
    E1 = [[96.  0.]]

✅ Added optimality cut to master problem.
optimal_cut : Size=3, Index={1, 2, 3}, Active=True
    Key : Lower  : Body                                           : Upper : Active
      1 : -520.0 :   0.0*x